In [1]:
import os
import pandas as pd
import numpy as np
import json


ocr_types_dict = {
    'Gemini-3-pro-preview':'Gemini3-pro-preview_demo_result_result',
}

result_folder = '../result'

# match_name = 'no_split'
match_name = 'quick_match'

In [2]:
# ====== Utility Functions ======
from __future__ import annotations

from pathlib import Path
from collections import defaultdict
import json
import re
import numpy as np
import pandas as pd
import argparse


def _page_id_from_indexed_key(key: str) -> str:
    """key 形如: xxx_[15] / xxx_[0]，返回 page_id=xxx。"""
    m = re.match(r"^(.*)_\[(\d+)\]$", key)
    return m.group(1) if m else key


def _first_existing(paths: list[Path]) -> Path:
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError("None exists: " + " | ".join(str(p) for p in paths))


def iter_json_array(path: Path, *, chunk_size: int = 1 << 20):
    """流式迭代一个巨大的 JSON 数组文件（避免一次性 json.load 占用内存）。"""
    decoder = json.JSONDecoder()
    with path.open("r", encoding="utf-8") as f:
        buf = ""
        # 定位到 '[' 之后
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                raise ValueError(f"Empty/invalid json array: {path}")
            buf += chunk
            i = buf.find("[")
            if i != -1:
                buf = buf[i + 1 :]
                break
        while True:
            buf = buf.lstrip()
            if not buf:
                chunk = f.read(chunk_size)
                if not chunk:
                    raise ValueError(f"Unexpected EOF in json array: {path}")
                buf += chunk
                continue
            if buf.startswith(","):
                buf = buf[1:]
                continue
            if buf.startswith("]"):
                return
            while True:
                try:
                    obj, end = decoder.raw_decode(buf)
                    break
                except json.JSONDecodeError:
                    chunk = f.read(chunk_size)
                    if not chunk:
                        raise
                    buf += chunk
            yield obj
            buf = buf[end:]


def load_text_scores_and_counts(*, result_folder: Path, ocr_prefix: str, split_tag: str = "All"):
    # 尝试加载对应 split 的 per_page_edit 文件
    # 如果找不到具体 split 的文件 (如 ..._digit_per_page_edit.json)，则回退到不带 split 的文件
    try:
        per_page_edit = _first_existing([
            result_folder / f"{ocr_prefix}_text_block_{split_tag}_per_page_edit.json",
            result_folder / f"{ocr_prefix}_text_block_per_page_edit.json",
        ])
    except FileNotFoundError:
        # 如果连通用文件都找不到，说明该模型可能没有 text 结果，返回空
        return pd.DataFrame(), pd.Series(dtype=int)

    text_block_result_path = _first_existing([
        result_folder / f"{ocr_prefix}_text_block_result.json",
    ])

    with per_page_edit.open("r", encoding="utf-8") as f:
        page_edit = json.load(f)
    
    # 如果文件是按 split 拆分的（文件名包含 split_tag），直接读取即可
    # 如果是回退到通用文件，我们需要手动过滤 keys (如果 page_edit 里包含所有图片)
    # 但通常 per_page_edit.json 只是一个 dict: {img_id: score}。
    # 更准确的方法是：通过 text_block_result.json 来判断哪些 img_id 属于当前 split，然后只保留这些 img_id 的分数。

    # n_text AND filter logic：按 img_id 统计 text_block 单元数，并确定该图片属于哪个 split
    want_is_digit: bool | None = None
    if split_tag == "digit":
        want_is_digit = True
    elif split_tag == "photo":
        want_is_digit = False
    
    valid_img_ids = set()
    counts = defaultdict(int)
    
    for item in iter_json_array(text_block_result_path):
        # 过滤 split
        if want_is_digit is not None:
            # 如果 item 中没有 is_digit 字段，默认视为 digit (True) 或跳过？
            # OmniDocBench 逻辑通常假定有该字段。如果没有，这里会报错或逻辑不符。
            # 这里宽松处理：如果没写 is_digit，默认当做 digit 处理 (True)
            is_orig = item.get("is_digit", True) 
            if bool(is_orig) != want_is_digit:
                continue
        
        img_id = item.get("img_id")
        if isinstance(img_id, str) and img_id:
            counts[img_id] += 1
            valid_img_ids.add(img_id)

    # 过滤 text_df，只保留属于当前 split 的 img_id
    # 注意：page_edit 里的 key 可能是 img_id
    filtered_page_edit = {k: v for k, v in page_edit.items() if k in valid_img_ids}
    
    if not filtered_page_edit:
        return pd.DataFrame(), pd.Series(dtype=int)

    text_df = pd.DataFrame({"img_id": list(filtered_page_edit.keys()), "text_edit": list(filtered_page_edit.values())})
    text_df["s_text"] = (1.0 - text_df["text_edit"].astype(float)).clip(0, 1)
    text_df = text_df.set_index("img_id")["s_text"].to_frame()

    n_text = pd.Series(counts, name="n_text").astype(int)
    # n_text 里可能包含 valid_img_ids 里有但 page_edit 里没有的（例如 edit 算失败了？），或者反之。
    # 取交集或以 valid_img_ids 为准
    n_text = n_text[n_text.index.isin(text_df.index)]

    return text_df, n_text


def load_formula_scores_and_counts(*, result_folder: Path, ocr_prefix: str, split_tag: str = "All"):
    try:
        per_sample_cdm = _first_existing([
            result_folder / f"{ocr_prefix}_display_formula_{split_tag}_per_sample_CDM.json",
            result_folder / f"{ocr_prefix}_display_formula_per_sample_CDM.json",
        ])
    except FileNotFoundError:
        return pd.DataFrame(), pd.Series(dtype=int)

    # 还需要 result.json 来过滤 is_digit
    formula_result_path = _first_existing([
        result_folder / f"{ocr_prefix}_display_formula_result.json",
    ])

    with per_sample_cdm.open("r", encoding="utf-8") as f:
        d = json.load(f)

    # 确定属于当前 split 的 valid_ids
    want_is_digit: bool | None = None
    if split_tag == "digit":
        want_is_digit = True
    elif split_tag == "photo":
        want_is_digit = False

    valid_keys = set()
    # formula_result.json 也是个大数组，包含 is_digit info
    # 注意：formula 的 result json 里 item 对应的是一个框 result，key 是 "img_id" + index? 
    # per_sample_CDM 的 key 是 "img_id_[idx]"
    # 我们遍历 result json 找到符合条件的 img_id 还是具体的 idx？
    # OmniDocBench 中 formula result 通常包含 "img_id" 和 "is_digit"
    
    # 建立 img_id -> is_digit 的映射 (假设一张图的 is_digit 是统一的)
    img_is_orig_map = {}
    
    # 同样使用流式读取，防止内存爆
    for item in iter_json_array(formula_result_path):
        img_id = item.get("img_id")
        if img_id and "is_digit" in item:
            img_is_orig_map[img_id] = item["is_digit"]
        elif img_id and img_id not in img_is_orig_map:
             # 默认 True
            img_is_orig_map[img_id] = True

    sums = defaultdict(float)
    cnts = defaultdict(int)
    
    for key, val in d.items():
        if val is None:
            continue
        page_id = _page_id_from_indexed_key(str(key))
        
        # 过滤
        if want_is_digit is not None:
            is_orig = img_is_orig_map.get(page_id, True) # 默认 True
            if bool(is_orig) != want_is_digit:
                continue

        sums[page_id] += float(val)
        cnts[page_id] += 1

    if not sums:
        return pd.DataFrame(), pd.Series(dtype=int)

    s_formula = pd.Series({k: sums[k] / cnts[k] for k in cnts}, name="s_formula").astype(float)
    n_formula = pd.Series(cnts, name="n_formula").astype(int)
    return s_formula.to_frame(), n_formula


def load_table_scores_and_counts(*, result_folder: Path, ocr_prefix: str, split_tag: str = "All", metric: str = "TEDS"):
    try:
        per_table = _first_existing([
            result_folder / f"{ocr_prefix}_table_{split_tag}_per_table_TEDS.json",
            result_folder / f"{ocr_prefix}_table_per_table_TEDS.json",
        ])
    except FileNotFoundError:
        return pd.DataFrame(), pd.Series(dtype=int)

    table_result_path = _first_existing([
        result_folder / f"{ocr_prefix}_table_result.json",
    ])
    
    # 确定过滤逻辑
    want_is_digit: bool | None = None
    if split_tag == "digit":
        want_is_digit = True
    elif split_tag == "photo":
        want_is_digit = False

    # 建立映射
    img_is_orig_map = {}
    for item in iter_json_array(table_result_path):
        img_id = item.get("img_id")
        if img_id:
             # 优先信 item 里的 is_digit，如果没有默认 True
            img_is_orig_map[img_id] = item.get("is_digit", True)

    with per_table.open("r", encoding="utf-8") as f:
        d = json.load(f)

    sums = defaultdict(float)
    cnts = defaultdict(int)
    for key, metrics in d.items():
        if not isinstance(metrics, dict):
            continue
        score = metrics.get(metric)
        if score is None:
            continue
        
        page_id = _page_id_from_indexed_key(str(key))
        
        # 过滤
        if want_is_digit is not None:
            is_orig = img_is_orig_map.get(page_id, True)
            if bool(is_orig) != want_is_digit:
                continue

        sums[page_id] += float(score)
        cnts[page_id] += 1

    if not sums:
        return pd.DataFrame(), pd.Series(dtype=int)

    s_table = pd.Series({k: sums[k] / cnts[k] for k in cnts}, name="s_table").astype(float)
    n_table = pd.Series(cnts, name="n_table").astype(int)
    return s_table.to_frame(), n_table


def lang_from_img_id(img_id: str) -> str:
    # 约定：img_id 形如 es_xxx / ar_xxx / zh_xxx ...，取第一个 '_' 前缀
    return img_id.split("_", 1)[0] if isinstance(img_id, str) and "_" in img_id else "unknown"


def compute_present_tasks_overall(pages: pd.DataFrame, *, empty_policy: str = "nan") -> pd.DataFrame:
    """方案二：每页 overall = 出现的任务类型基础分数的均值。"""
    if empty_policy not in {"nan", "zero"}:
        raise ValueError("empty_policy must be 'nan' or 'zero'")

    df = pages.copy()

    for k in ["text", "formula", "table"]:
        df[f"n_{k}"] = df[f"n_{k}"].fillna(0).astype(float) if f"n_{k}" in df else 0.0
        df[f"s_{k}"] = df[f"s_{k}"].astype(float) if f"s_{k}" in df else np.nan

    use_text = (df["n_text"] > 0) & df["s_text"].notna()
    use_formula = (df["n_formula"] > 0) & df["s_formula"].notna()
    use_table = (df["n_table"] > 0) & df["s_table"].notna()

    denom = use_text.astype(int) + use_formula.astype(int) + use_table.astype(int)
    numer = (
        df["s_text"].fillna(0) * use_text.astype(int)
        + df["s_formula"].fillna(0) * use_formula.astype(int)
        + df["s_table"].fillna(0) * use_table.astype(int)
    )

    overall_i = numer / denom.replace({0: np.nan})
    if empty_policy == "zero":
        overall_i = overall_i.fillna(0.0)

    df["overall_i_scheme2"] = overall_i.clip(0, 1)
    df["present_task_cnt"] = denom
    df["has_text"] = use_text
    df["has_formula"] = use_formula
    df["has_table"] = use_table
    return df


def count_dataset_pages_from_md(prefix: str) -> int | None:
    # Try OmniDocBench result folder (../result)
    start_paths = [Path("../result")]
    for root in start_paths:
        d = root / prefix
        if d.exists():
            return sum(1 for _ in d.rglob("*.md"))
    return None

In [3]:
# ====== 参数 ======

result_folder = Path("../result")
match_name = "quick_match"
# 定义要计算的 split tag，包括 'All' 以便在 Overall 列中显示
split_tags = ["All", "digit", "photo"] 

split_rename_map = {"digit": "Digit.", "photo": "Photo.", "All": "All"}

latin_langs = ["DE", "EN", "ES", "FR", "ID", "IT", "NL", "PT", "VI"]
non_latin_langs = ["AR", "HI", "JP", "KO", "RU", "TH", "ZH", "ZH-T"]

summary_rows = []

for model_name, prefix in ocr_types_dict.items():
    model_data = {}  # Dictionary to store scores
    
    for split in split_tags:
        try:
            # Load scores and counts for the current split
            text_s, n_text = load_text_scores_and_counts(
                result_folder=Path(result_folder) / prefix, 
                ocr_prefix=prefix, 
                
                split_tag=split
            )
            formula_s, n_formula = load_formula_scores_and_counts(
                result_folder=Path(result_folder) / prefix, 
                ocr_prefix=prefix, 
                
                split_tag=split
            )
            table_s, n_table = load_table_scores_and_counts(
                result_folder=Path(result_folder) / prefix, 
                ocr_prefix=prefix, 
                
                split_tag=split, 
                metric="TEDS"
            )

            # Combine data
            pages = (
                text_s
                .join(pd.Series(n_text, name="n_text"), how="outer")
                .join(pd.Series(formula_s.iloc[:,0] if not formula_s.empty else [], name="s_formula"), how="outer")
                .join(pd.Series(n_formula, name="n_formula"), how="outer")
                .join(pd.Series(table_s.iloc[:,0] if not table_s.empty else [], name="s_table"), how="outer")
                .join(pd.Series(n_table, name="n_table"), how="outer")
            )
            
            if len(pages) > 0:
                pages.index.name = "img_id"
                pages["lang"] = [lang_from_img_id(i) for i in pages.index]

                # Compute overall score (Scheme 2)
                pages_scheme2 = compute_present_tasks_overall(pages, empty_policy="nan")
                
                # 1. Group by Language (Only for All to get combined total score)
                if split == "All":
                    lang_group = pages_scheme2.groupby("lang")["overall_i_scheme2"].mean()
                    for lang, score in lang_group.items():
                        lang_up = lang.upper()
                        if lang_up == "ZH-CHT":
                            lang_up = "ZH-T"
                        model_data[(lang_up, "")] = round(score * 100, 1)
                
                # 2. Compute Overall (across all languages for THIS split)
                overall_score = pages_scheme2["overall_i_scheme2"].mean(skipna=True)
                model_data[("Overall", split_rename_map[split])] = round(overall_score * 100, 1)
            else:
                pass

        except Exception as e:
            print(f"Error processing {model_name} with split {split}: {e}")
    
    # Compute Latin Avg
    latin_scores = [model_data[(l, "")] for l in latin_langs if (l, "") in model_data]
    if latin_scores:
        model_data[("Latin_Avg", "")] = round(sum(latin_scores) / len(latin_scores), 1)
        
    # Compute Non-Latin Avg
    non_latin_scores = [model_data[(l, "")] for l in non_latin_langs if (l, "") in model_data]
    if non_latin_scores:
        model_data[("Non-Latin_Avg", "")] = round(sum(non_latin_scores) / len(non_latin_scores), 1)

    # Store the row data
    row_entry = {"OCR_Model": model_name}
    for l in latin_langs:
        row_entry[("Latin", l)] = model_data.get((l, ""), None)
    row_entry[("Latin", "Avg")] = model_data.get(("Latin_Avg", ""), None)

    for l in non_latin_langs:
        row_entry[("Non-Latin", l)] = model_data.get((l, ""), None)
    row_entry[("Non-Latin", "Avg")] = model_data.get(("Non-Latin_Avg", ""), None)

    for s in ["All", "Digit.", "Photo."]:
        row_entry[("Overall", s)] = model_data.get(("Overall", s), None)

    summary_rows.append(row_entry)

# Create DataFrame
df_summary = pd.DataFrame(summary_rows).set_index("OCR_Model")

# Create MultiIndex columns
df_summary.columns = pd.MultiIndex.from_tuples(df_summary.columns, names=["Category", "Language/Split"])


print("Summary Table:")
display(df_summary)

print('\nLaTeX Code:')
print(df_summary.to_latex())


Summary Table:


Category             Latin                                                  \
Language/Split          DE    EN    ES    FR    ID    IT    NL    PT    VI   
OCR_Model                                                                    
Gemini-3-pro-preview  65.9  92.9  25.7  91.9  93.0  97.6  84.4  98.4  90.6   

Category                    ... Non-Latin                                     \
Language/Split         Avg  ...        JP    KO    RU    TH    ZH ZH-T   Avg   
OCR_Model                   ...                                                
Gemini-3-pro-preview  82.3  ...      65.7  91.3  97.4  97.2  99.2  4.7  79.7   

Category             Overall                
Language/Split           All Digit. Photo.  
OCR_Model                                   
Gemini-3-pro-preview    82.2   94.5   78.3  

[1 rows x 22 columns]


LaTeX Code:
\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrr}
\toprule
Category & \multicolumn{10}{r}{Latin} & \multicolumn{9}{r}{Non-Latin} & \multicolumn{3}{r}{Overall} \\
Language/Split & DE & EN & ES & FR & ID & IT & NL & PT & VI & Avg & AR & HI & JP & KO & RU & TH & ZH & ZH-T & Avg & All & Digit. & Photo. \\
OCR_Model &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
Gemini-3-pro-preview & 65.900000 & 92.900000 & 25.700000 & 91.900000 & 93.000000 & 97.600000 & 84.400000 & 98.400000 & 90.600000 & 82.300000 & 83.400000 & 98.300000 & 65.700000 & 91.300000 & 97.400000 & 97.200000 & 99.200000 & 4.700000 & 79.700000 & 82.200000 & 94.500000 & 78.300000 \\
\bottomrule
\end{tabular}



In [18]:
# ====== 保存结果 ======

# 1. 保存为 CSV 文件
csv_filename = "summary_table.csv"
df_summary.to_csv(csv_filename)
print(f"表格已保存到: {csv_filename}")

# 2. 保存为 Markdown 文件 (方便查看)
md_filename = "summary_table.md"
with open(md_filename, "w", encoding="utf-8") as f:
    f.write(df_summary.to_markdown())
print(f"表格已保存到: {md_filename}")

# 3. 保存为 Excel 文件 (需要安装 openpyxl: pip install openpyxl)
try:
    xlsx_filename = "summary_table.xlsx"
    df_summary.to_excel(xlsx_filename)
    print(f"表格已保存到: {xlsx_filename}")
except ImportError:
    print("未安装 openpyxl，跳过 Excel 保存。如需保存为 Excel，请运行 `pip install openpyxl`")
except Exception as e:
    print(f"保存 Excel 时出错: {e}")

表格已保存到: summary_table.csv
表格已保存到: summary_table.md
未安装 openpyxl，跳过 Excel 保存。如需保存为 Excel，请运行 `pip install openpyxl`


In [5]:
# ====== 整体汇总是（ALL, digit, photo） ======
# 不分语言，统计所有样本在不同 Split 下的 Overall 得分

# 定义需要计算的 split tag
split_tags = ["All", "digit", "photo"]
summary_split_list = []

print("Running Overall Calculation Across All Languages...")

for model_name, prefix in ocr_types_dict.items():
    row_data = {"OCR_Model": model_name}
    
    for split_tag in split_tags:
        # Load scores and counts
        try:
            # 加载数据，这里复用 cell 1 / cell 4 中出现过的 helper 函数
            # 注意：load_xxx_scores_and_counts 需要在前面的 cell 中已经定义并运行过
            text_s, n_text = load_text_scores_and_counts(
                result_folder=Path(result_folder) / prefix, 
                ocr_prefix=prefix, 
                
                split_tag=split_tag
            )
            formula_s, n_formula = load_formula_scores_and_counts(
                result_folder=Path(result_folder) / prefix, 
                ocr_prefix=prefix, 
                
                split_tag=split_tag
            )
            table_s, n_table = load_table_scores_and_counts(
                result_folder=Path(result_folder) / prefix, 
                ocr_prefix=prefix, 
                
                split_tag=split_tag, 
                metric="TEDS"
            )

            # 合并所有任务数据
            pages = (
                text_s
                .join(pd.Series(n_text, name="n_text"), how="outer")
                .join(pd.Series(formula_s.iloc[:,0] if not formula_s.empty else [], name="s_formula"), how="outer")
                .join(pd.Series(n_formula, name="n_formula"), how="outer")
                .join(pd.Series(table_s.iloc[:,0] if not table_s.empty else [], name="s_table"), how="outer")
                .join(pd.Series(n_table, name="n_table"), how="outer")
            )
            
            # 计算 Scheme 2 Overall
            if len(pages) > 0:
                pages_scheme2 = compute_present_tasks_overall(pages, empty_policy="nan")
                overall_val = pages_scheme2["overall_i_scheme2"].mean(skipna=True)
                # 存百分比
                row_data[split_tag] = round(overall_val * 100, 3)
            else:
                row_data[split_tag] = None
        except Exception as e:
            print(f"Error processing {model_name} with split {split_tag}: {e}")
            row_data[split_tag] = None

    summary_split_list.append(row_data)

print("\nOverall Scores by Split (All/digit/photo) - Across All Languages:")
df_split_summary = pd.DataFrame(summary_split_list).set_index("OCR_Model")
# 调整列顺序
df_split_summary = df_split_summary[["All", "digit", "photo"]]
display(df_split_summary)

Running Overall Calculation Across All Languages...



Overall Scores by Split (All/digit/photo) - Across All Languages:


,All,digit,photo
OCR_Model,,,
Gemini-3-pro-preview,81.201,94.205,77.137


In [6]:
# ====== 公式分数表（按单元格三格式：行=Language/Split，列=Model） ======
from pathlib import Path
import json
import numpy as np
import pandas as pd


def _normalize_lang_key_for_table(lang_key: str) -> str:
    if not isinstance(lang_key, str):
        return str(lang_key)
    return lang_key.replace("language: ", "").strip()


def build_split_metric_table(
    *,
    category_type: str,
    metric_name: str,
    metric_mode: str = "score",  # score: 越大越好(0-1), edit_dist: 越小越好(0-1)
    split_tags: list[str] | None = None,
) -> pd.DataFrame:
    if split_tags is None:
        split_tags = ["All", "digit", "photo"]

    # 17种指定的语言及其指定顺序
    # 注意：对应到你 json 里提取出的 language 完整名称（这里作小写匹配归一化处理）
    # ar -> arabic, de -> german, en -> english, es -> spanish, fr -> french
    # hi -> hindi, id -> indonesian, it -> italian, jp -> japanese, ko -> korean
    # nl -> dutch, pt -> portuguese, ru -> russian, th -> thai
    # vi -> vietnamese, zh -> simplified_chinese, zh-CHT -> traditional-chinese
    ordered_target_langs = [
        "arabic", "german", "english", "spanish", "french", "hindi", 
        "indonesian", "italian", "japanese", "korean", "dutch", 
        "portuguese", "russian", "thai", "vietnamese", 
        "simplified_chinese", "traditional-chinese"
    ]
    target_langs = set(ordered_target_langs)

    rows = []
    for model_name, prefix in ocr_types_dict.items():
        model_data = {}
        result_path = Path(result_folder) / prefix / f"{prefix}_metric_result.json"

        if not result_path.exists():
            print(f"[Skip] 找不到文件: {result_path}")
            rows.append({"OCR_Model": model_name})
            continue

        with result_path.open("r", encoding="utf-8") as f:
            result = json.load(f)

        for split in split_tags:
            key_suffix = "" if split == "All" else f"_{split}"
            page_key = f"page{key_suffix}"

            metric_dict = (((result.get(category_type, {}) or {}).get(page_key, {}) or {}).get(metric_name, {}))
            if not isinstance(metric_dict, dict) or len(metric_dict) == 0:
                continue

            for lang_key, raw_val in metric_dict.items():
                if raw_val is None:
                    continue
                try:
                    v = float(raw_val)
                except Exception:
                    continue

                score = (1.0 - v) if metric_mode == "edit_dist" else v
                score = max(0.0, min(1.0, score))
                score_pct = round(score * 100, 1)

                if str(lang_key).upper() == "ALL":
                    model_data[("Overall", split)] = score_pct
                else:
                    lang = _normalize_lang_key_for_table(str(lang_key))
                    # 仅保留这 17 种语言
                    if lang.lower() in target_langs:
                        if split in ["digit", "photo"]:
                            model_data[(lang, split)] = score_pct

        row_entry = {col: val for col, val in model_data.items()}
        row_entry["OCR_Model"] = model_name
        rows.append(row_entry)

    df_summary = pd.DataFrame(rows).set_index("OCR_Model")
    if df_summary.empty:
        return df_summary

    metric_cols = [c for c in df_summary.columns if isinstance(c, tuple) and len(c) == 2]
    if len(metric_cols) == 0:
        return pd.DataFrame(index=pd.Index([], name="Language/Split"))

    df_summary = df_summary[metric_cols]
    df_summary.columns = pd.MultiIndex.from_tuples(df_summary.columns, names=["Language", "Split"])

    # 从当前 DataFrame 里提取真实大小写的语言名，映射成小写方便比对
    existing_langs = list(df_summary.columns.get_level_values(0).unique())
    existing_langs_lower_map = {l.lower(): l for l in existing_langs if l != "Overall"}

    new_columns = []
    
    # 按照要求的语言顺序添加
    for expected_lang_lower in ordered_target_langs:
        if expected_lang_lower in existing_langs_lower_map:
            actual_lang = existing_langs_lower_map[expected_lang_lower]
            for split in ["digit", "photo"]:
                if (actual_lang, split) in df_summary.columns:
                    new_columns.append((actual_lang, split))

    # 最后添加 Overall
    for split in ["All", "digit", "photo"]:
        if ("Overall", split) in df_summary.columns:
            new_columns.append(("Overall", split))

    if len(new_columns) == 0:
        return pd.DataFrame(index=pd.Index([], name="Language/Split"))

    df_summary = df_summary.reindex(columns=pd.MultiIndex.from_tuples(new_columns))
    return df_summary.T


df_formula_split = build_split_metric_table(
    category_type="display_formula",
    metric_name="CDM",
    metric_mode="score",
)

print("\n公式分数表（Rows=Lang/Split, Cols=Models）:")
df_formula_split.to_csv("formula_summary.csv")
print("已保存到 formula_summary.csv")
display(df_formula_split)


公式分数表（Rows=Lang/Split, Cols=Models）:
已保存到 formula_summary.csv


""
OCR_Model
Gemini-3-pro-preview


In [7]:
# ====== 文本分数表（按单元格三格式：行=Language/Split，列=Model） ======
# 说明：text_block 使用 Edit_dist，先做 1-Edit_dist 再转百分比

if "build_split_metric_table" not in globals():
    raise RuntimeError("请先运行上一个单元（定义 build_split_metric_table）")

df_text_split = build_split_metric_table(
    category_type="text_block",
    metric_name="Edit_dist",
    metric_mode="edit_dist",
)

print("\n文本分数表（Rows=Lang/Split, Cols=Models）:")
df_text_split.to_csv("text_summary.csv")
print("已保存到 text_summary.csv")
display(df_text_split)


文本分数表（Rows=Lang/Split, Cols=Models）:
已保存到 text_summary.csv


OCR_Model                  Gemini-3-pro-preview
Arabic              digit                  94.5
                    photo                  76.4
German              digit                  99.7
                    photo                  99.0
english             digit                  99.5
                    photo                  99.5
Spanish             digit                  99.8
                    photo                   1.0
French              digit                  92.8
                    photo                  91.9
Hindi               digit                  99.2
                    photo                  98.2
Indonesian          digit                  97.7
                    photo                  97.9
Japanese            digit                  91.6
                    photo                  33.1
Korean              digit                  97.1
                    photo                  97.0
Dutch               digit                  98.5
                    photo                  79.8
Portuguese          digit                  99.8
                    photo                  97.9
Russian             digit                 100.0
                    photo                  96.5
Thai                digit                 100.0
                    photo                  99.9
Vietnamese          digit                  90.4
                    photo                  91.4
simplified_chinese  digit                 100.0
                    photo                  99.0
Traditional-Chinese photo                   4.9
Overall             All                    82.3
                    digit                  97.4
                    photo                  78.8

In [8]:
# ====== 表格分数表（按单元格三格式：行=Language/Split，列=Model） ======
# 说明：table 使用 TEDS（0-1），直接转百分比

if "build_split_metric_table" not in globals():
    raise RuntimeError("请先运行上一个单元（定义 build_split_metric_table）")

df_table_split = build_split_metric_table(
    category_type="table",
    metric_name="TEDS",
    metric_mode="score",
)

print("\n表格分数表（Rows=Lang/Split, Cols=Models）:")
df_table_split.to_csv("table_summary.csv")
print("已保存到 table_summary.csv")
display(df_table_split)


表格分数表（Rows=Lang/Split, Cols=Models）:
已保存到 table_summary.csv


OCR_Model         Gemini-3-pro-preview
German     digit                  36.9
           photo                  32.9
english    digit                  85.3
           photo                  86.6
Hindi      digit                  99.2
           photo                  98.7
Indonesian digit                  88.1
           photo                  88.0
Japanese   digit                  98.2
           photo                  78.6
Korean     digit                  89.8
           photo                  86.4
Thai       digit                 100.0
           photo                  92.8
Overall    All                    83.4
           digit                  85.4
           photo                  81.1

In [9]:
# ====== 阅读顺序分数表（按单元格三格式：行=Language/Split，列=Model） ======
# 说明：reading_order 使用 Edit_dist，先做 1-Edit_dist 再转百分比

if "build_split_metric_table" not in globals():
    raise RuntimeError("请先运行上一个单元（定义 build_split_metric_table）")

df_reading_order_split = build_split_metric_table(
    category_type="reading_order",
    metric_name="Edit_dist",
    metric_mode="edit_dist",
)

print("\n阅读顺序分数表（Rows=Lang/Split, Cols=Models）:")
df_reading_order_split.to_csv("reading_order_summary.csv")
print("已保存到 reading_order_summary.csv")
display(df_reading_order_split)


阅读顺序分数表（Rows=Lang/Split, Cols=Models）:
已保存到 reading_order_summary.csv


OCR_Model                  Gemini-3-pro-preview
Arabic              digit                  79.3
                    photo                  78.4
German              digit                 100.0
                    photo                 100.0
english             digit                 100.0
                    photo                 100.0
Spanish             digit                 100.0
                    photo                   0.0
French              digit                  90.9
                    photo                  82.6
Hindi               digit                 100.0
                    photo                 100.0
Indonesian          digit                 100.0
                    photo                 100.0
Japanese            digit                  82.7
                    photo                  50.6
Korean              digit                 100.0
                    photo                 100.0
Dutch               digit                  57.9
                    photo                  56.1
Portuguese          digit                  66.7
                    photo                  66.7
Russian             digit                 100.0
                    photo                 100.0
Thai                digit                 100.0
                    photo                 100.0
Vietnamese          digit                  81.8
                    photo                  79.5
simplified_chinese  digit                 100.0
                    photo                 100.0
Traditional-Chinese photo                   9.4
Overall             All                    78.3
                    digit                  90.6
                    photo                  75.6